### Zach Burgess, Brandon Lowery, Jason Martin
### COSC 526 Data Engineering
### Dr. Linder

In [25]:
# stage 0: import data
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataImport").getOrCreate()

df = spark.read.csv('train.csv', header=True, inferSchema=True, multiLine=True, escape='"')

In [26]:
# stage 1: short EDA
from pyspark.sql import functions as F

df.printSchema()

print(df.dtypes)

print(df.columns)

df.describe().show()

df.summary().show()

print('null check:')
df.select([F.count(F.when(df[c].isNull(), c)).alias(c) for c in df.columns]).show()

df.show(5, truncate=True)

root
 |-- id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- author: string (nullable = true)

[('id', 'string'), ('text', 'string'), ('author', 'string')]
['id', 'text', 'author']
+-------+-------+--------------------+------+
|summary|     id|                text|author|
+-------+-------+--------------------+------+
|  count|  19579|               19579| 19579|
|   mean|   NULL|                NULL|  NULL|
| stddev|   NULL|                NULL|  NULL|
|    min|id00001|" Odenheimer, res...|   EAP|
|    max|id27971|you could not hop...|   MWS|
+-------+-------+--------------------+------+

+-------+-------+--------------------+------+
|summary|     id|                text|author|
+-------+-------+--------------------+------+
|  count|  19579|               19579| 19579|
|   mean|   NULL|                NULL|  NULL|
| stddev|   NULL|                NULL|  NULL|
|    min|id00001|" Odenheimer, res...|   EAP|
|    25%|   NULL|                NULL|  NULL|
|    50%|   NULL|

In [27]:
# stage 1: plot EDA
df2 = df.groupBy('author').count()
df2.plot.bar(x='author', y='count')

In [28]:
# stage 1: data prep
from pyspark.ml.feature import StopWordsRemover, Tokenizer

tokenizer = Tokenizer(inputCol='text', outputCol='words')
stop_word_remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')

In [29]:
# stage 2: feature extraction
from pyspark.ml.feature import CountVectorizer, IDF, Normalizer

count_vectorizer = CountVectorizer(inputCol='filtered_words', outputCol='vectorized_features')
idf = IDF(inputCol='vectorized_features', outputCol='features')
normalizer = Normalizer(inputCol='features', outputCol='normalized_features')

In [30]:
# build pipeline and run
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[tokenizer, stop_word_remover, count_vectorizer, idf, normalizer])
final_df = pipeline.fit(df).transform(df)

In [31]:
# confirm

final_df.printSchema()

print(final_df.dtypes)

print(final_df.columns)

print(final_df.describe().show())

print(final_df.summary().show())

final_df.show(5, truncate=True)

root
 |-- id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- author: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- vectorized_features: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- normalized_features: vector (nullable = true)

[('id', 'string'), ('text', 'string'), ('author', 'string'), ('words', 'array<string>'), ('filtered_words', 'array<string>'), ('vectorized_features', 'vector'), ('features', 'vector'), ('normalized_features', 'vector')]
['id', 'text', 'author', 'words', 'filtered_words', 'vectorized_features', 'features', 'normalized_features']
+-------+-------+--------------------+------+
|summary|     id|                text|author|
+-------+-------+--------------------+------+
|  count|  19579|               19579| 19579|
|   mean|   NULL|                NULL|  NULL|
| stdde